# Splitting strategy comparison

Compares two chunking strategies for the HP RAG pipeline:

- **Strategy A — token-based fixed-length** (current): `RecursiveCharacterTextSplitter.from_tiktoken_encoder`, 500 tokens, 50-token overlap. Guarantees a tight token budget per chunk but can cut mid-sentence.
- **Strategy B — character-based recursive**: `RecursiveCharacterTextSplitter`, ~2 000 characters, 200-character overlap, default separators `["\n\n", "\n", ". ", " ", ""]`. Splits on natural boundaries (paragraphs → sentences → words) before falling back to characters.

Sections:
1. Chunk size distributions
2. Sentence boundary quality
3. Sample chunk inspection
4. Retrieval quality on HP test queries

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tiktoken
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [ ]:
PDF_PATH   = "harrypotter.pdf"
EMBED_MODEL = "all-MiniLM-L6-v2"

docs = PyMuPDFLoader(PDF_PATH).load()
print(f"Loaded {len(docs)} pages")

# Strategy A: token-based fixed-length (current production setting)
splitter_A = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50,
)
chunks_A = splitter_A.split_documents(docs)

# Strategy B: character-based recursive (splits on natural boundaries first)
splitter_B = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # ~500 tokens at ~4 chars/token
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_B = splitter_B.split_documents(docs)

print(f"Strategy A (token-based):  {len(chunks_A):,} chunks")
print(f"Strategy B (char-based):   {len(chunks_B):,} chunks")

## 1. Chunk size distributions

Token-based splitting produces a tight spike at 500 tokens. Character-based splitting produces a wider distribution because it respects natural text boundaries — some paragraphs are short, others long.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

tok_A = [len(enc.encode(c.page_content)) for c in chunks_A]
tok_B = [len(enc.encode(c.page_content)) for c in chunks_B]
chr_A = [len(c.page_content) for c in chunks_A]
chr_B = [len(c.page_content) for c in chunks_B]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, data_A, data_B, xlabel, title in [
    (axes[0], tok_A, tok_B, "Tokens", "Token length distribution"),
    (axes[1], chr_A, chr_B, "Characters", "Character length distribution"),
]:
    ax.hist(data_A, bins=60, alpha=0.6, label="A: token-based", color="steelblue")
    ax.hist(data_B, bins=60, alpha=0.6, label="B: char-based", color="coral")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of chunks")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

for label, data in [("A token-based", tok_A), ("B char-based", tok_B)]:
    print(f"{label:20s}  mean={np.mean(data):5.0f}  std={np.std(data):5.0f}"
          f"  min={min(data):4d}  max={max(data):4d}  chunks={len(data):,}")

## 2. Sentence boundary quality

A good chunk should start at a sentence or paragraph boundary and end at one too. Mid-sentence splits force the LLM to reason over incomplete context.

- **Ends at sentence boundary**: last non-whitespace character is `.`, `!`, `?`, `"`, or `'`
- **Starts mid-sentence**: first non-whitespace character is lowercase (continuation of a previous sentence)

In [ ]:
def boundary_stats(chunks):
    ends_sentence, starts_mid = 0, 0
    for c in chunks:
        text = c.page_content
        if text.rstrip() and text.rstrip()[-1] in ".!?\"'":
            ends_sentence += 1
        if text.lstrip() and text.lstrip()[0].islower():
            starts_mid += 1
    n = len(chunks)
    return ends_sentence / n * 100, starts_mid / n * 100

end_A, mid_A = boundary_stats(chunks_A)
end_B, mid_B = boundary_stats(chunks_B)

labels   = ["Ends at sentence boundary ↑", "Starts mid-sentence ↓"]
vals_A   = [end_A, mid_A]
vals_B   = [end_B, mid_B]
colors_A = ["steelblue", "steelblue"]
colors_B = ["coral", "coral"]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars_A = ax.bar(x - w/2, vals_A, w, label="A: token-based", color="steelblue", alpha=0.8)
bars_B = ax.bar(x + w/2, vals_B, w, label="B: char-based",  color="coral",     alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("% of chunks")
ax.set_title("Chunk boundary quality")
ax.set_ylim(0, 100)
ax.legend()
for bar in list(bars_A) + list(bars_B):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Sample chunk inspection

Side-by-side view of the same 3 random positions in the corpus. Pay attention to whether each chunk opens and closes on a complete thought.

In [ ]:
random.seed(42)
PREVIEW = 350  # characters to show per chunk

sample_ids = random.sample(range(min(len(chunks_A), len(chunks_B))), 3)

for i, idx in enumerate(sample_ids):
    ca, cb = chunks_A[idx], chunks_B[idx]
    print(f"{'═'*72}")
    print(f"  SAMPLE {i+1}  —  position #{idx} in corpus")
    print(f"{'─'*72}")
    print(f"  A: token-based   {tok_A[idx]:3d} tok  {chr_A[idx]:4d} chars")
    print(f"  {ca.page_content[:PREVIEW].strip()!r}")
    print()
    print(f"  B: char-based    {tok_B[idx]:3d} tok  {chr_B[idx]:4d} chars")
    print(f"  {cb.page_content[:PREVIEW].strip()!r}")
    print()

## 4. Retrieval quality

For a set of HP questions, embed the query and retrieve the top-3 most similar chunks from each strategy. Better splitting should return chunks that contain the answer in a readable, complete context — not a fragment that starts or ends mid-sentence.

In [ ]:
# Embedding all chunks takes a few minutes on CPU
model = SentenceTransformer(EMBED_MODEL)

texts_A = [c.page_content for c in chunks_A]
texts_B = [c.page_content for c in chunks_B]

print("Embedding strategy A...")
embs_A = model.encode(texts_A, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
print("Embedding strategy B...")
embs_B = model.encode(texts_B, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
print("Done.")

In [ ]:
TEST_QUERIES = [
    "What is the name of Harry's owl?",
    "Who is the Defence Against the Dark Arts teacher in year one?",
    "What are the three Deathly Hallows?",
    "Where is the entrance to the Chamber of Secrets?",
    "How does Harry destroy the Horcrux in the diary?",
]

CONTEXT_CHARS = 400  # characters of each retrieved chunk to display

def top_k(query, embs, texts, k=3):
    q = model.encode([query], normalize_embeddings=True)
    scores = (embs @ q.T).squeeze()
    idx = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), texts[i]) for i in idx]

for query in TEST_QUERIES:
    res_A = top_k(query, embs_A, texts_A)
    res_B = top_k(query, embs_B, texts_B)
    print(f"\n{'═'*72}")
    print(f"  QUERY: {query}")
    for label, results in [("A token-based", res_A), ("B char-based", res_B)]:
        score, chunk = results[0]
        print(f"\n  [{label}]  score={score:.3f}")
        print(f"  {chunk[:CONTEXT_CHARS].strip()!r}")

## 5. Retrieval score distribution

For each query, plot the top-20 cosine similarity scores from both strategies. A tighter gap between rank-1 and rank-2 scores suggests ambiguity; a sharp drop-off means the top chunk is clearly the best match.

In [ ]:
K_PLOT = 20

fig, axes = plt.subplots(len(TEST_QUERIES), 1, figsize=(12, 3 * len(TEST_QUERIES)), sharex=True)

for ax, query in zip(axes, TEST_QUERIES):
    for label, embs, color in [
        ("A: token-based", embs_A, "steelblue"),
        ("B: char-based",  embs_B, "coral"),
    ]:
        q = model.encode([query], normalize_embeddings=True)
        scores = np.sort((embs @ q.T).squeeze())[::-1][:K_PLOT]
        ax.plot(range(1, K_PLOT + 1), scores, marker="o", markersize=4,
                label=label, color=color, linewidth=1.5)
    ax.set_title(query, fontsize=9)
    ax.set_ylabel("cosine sim")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Rank")
plt.suptitle("Top-20 retrieval scores per query", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()